# Yield Curve Dynamics — Colab Training

Run cells top-to-bottom.

**Runtime:** `Runtime > Change runtime type > GPU` (recommended for Stage B constraints)

**Note:** Raw/processed data is **not** in GitHub (gitignored). This notebook downloads FRED data and preprocesses it automatically on first run.

In [ ]:
# Clone repo (skip if you already uploaded the project)
import os

REPO_URL = "https://github.com/danield116/Yield-Curve-Dynamics-ML.git"
REPO_DIR = "/content/Yield-Curve-Dynamics-ML"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

PROJECT_DIR = f"{REPO_DIR}/yield-curve-geometric-sde"
%cd {PROJECT_DIR}
print("Working directory:", os.getcwd())

In [ ]:
# Install Python dependencies (torch is usually preinstalled on Colab)
!pip install -q pyyaml pandas numpy scipy scikit-learn matplotlib seaborn tqdm statsmodels

In [ ]:
# Optional: mount Google Drive to persist checkpoints across sessions
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/yield-curve-geometric-sde"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Drive root:", DRIVE_ROOT)

In [ ]:
# Download FRED yields + preprocess (creates data/raw and data/processed)
import os
from pathlib import Path

# Colab resets cwd on reconnect — always jump back to project root.
PROJECT_DIR = "/content/Yield-Curve-Dynamics-ML/yield-curve-geometric-sde"
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

!python data/download_fred_yields.py \
  --start-date 2001-07-01 \
  --output-path data/raw/fred_yields.csv

!python data/preprocess_curves.py \
  --input-path data/raw/fred_yields.csv \
  --output-dir data/processed \
  --levelscript

processed = Path("data/processed")
expected = ["train_scaled.csv", "val_scaled.csv", "test_scaled.csv"]
missing = [name for name in expected if not (processed / name).exists()]
if missing:
    raise FileNotFoundError(
        f"Preprocess did not create: {missing}. "
        "Scroll up in this cell for download/preprocess errors, then re-run."
    )
print("Processed files OK:", [f.name for f in sorted(processed.glob("*.csv"))])

In [ ]:
# Quick sanity check: data shapes + split plot
import os
import pandas as pd
from pathlib import Path

PROJECT_DIR = "/content/Yield-Curve-Dynamics-ML/yield-curve-geometric-sde"
os.chdir(PROJECT_DIR)

processed = Path("data/processed")
required = ["train_scaled.csv", "val_scaled.csv", "test_scaled.csv"]
missing = [name for name in required if not (processed / name).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} under {processed.resolve()}. "
        "Run the previous cell (FRED download + preprocess) first and fix any errors there."
    )

train = pd.read_csv(processed / "train_scaled.csv", index_col=0, parse_dates=True)
val = pd.read_csv(processed / "val_scaled.csv", index_col=0, parse_dates=True)
test = pd.read_csv(processed / "test_scaled.csv", index_col=0, parse_dates=True)

print(f"train: {train.shape} | val: {val.shape} | test: {test.shape}")
print(f"train dates: {train.index.min().date()} -> {train.index.max().date()}")

!python data/visualize_splits.py \
  --processed-dir data/processed \
  --suffix scaled \
  --output-path reports/figures/split_boundaries_scaled.png

In [ ]:
# Stage A: train manifold model (Student-t CVAE by default)
import torch
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

!python training/train_stage_a.py --config config/default.yaml

In [ ]:
# Stage B ablations (run all 4 constraint paths)
ablations = ["sde_only", "sde_pde", "sde_jacobian", "sde_both"]

for ablation in ablations:
    print("\n" + "=" * 60)
    print(f"Training Stage B ablation: {ablation}")
    print("=" * 60)
    !python training/train_stage_b.py --config config/default.yaml --ablation {ablation}

In [ ]:
# Optional: copy artifacts to Google Drive
import shutil

for folder in ["reports/checkpoints", "reports/latents", "reports/forecasts", "reports/figures"]:
    src = Path(folder)
    if src.exists():
        dst = Path(DRIVE_ROOT) / folder
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"Copied {src} -> {dst}")

print("Done. Checkpoints and forecasts saved to Drive.")